# Run N: GCNet-S (Golden Cudgel Network), binary defect segmentation

GCNet was published at CVPR 2025: *Golden Cudgel Network for Real-Time Semantic Segmentation* (arXiv:2503.03325). It's a reparameterizable dual-branch (semantic + detail) real-time segmentation network: each conv block trains as 3 parallel paths (two 3x3 + one 1x1 conv, RepVGG-style) that would be fused into a single 3x3 conv for deployment -- "self-enlarge during training, self-contract during inference." This notebook only needs the training-time (multi-branch) form, so the fusion/`switch_to_deploy` step (an inference-speed optimization that doesn't change any metric this project reports) is intentionally not ported.

The official implementation (`gyyang23/GCNet`) is a full vendored copy of MMSegmentation -- this notebook instead ports just the architecture-defining files (`backbones/gcnet.py`, `decode_heads/gcnet_head.py`, the `DAPPM` block from `utils/ppm.py`) as plain PyTorch, with `mmcv.ConvModule` / `mmengine.BaseModule` swapped for direct equivalents (forward-pass math unchanged). Verified locally before ever touching Kaggle: dummy forward+backward gives correct shapes and 100% gradient coverage, and the real GCNet-S Cityscapes checkpoint's `backbone.*` weights match the vendored backbone with **0 missing / 0 unexpected keys**; 22 of 26 decode-head tensors also transfer (the 4 dropped ones are the Cityscapes 19-class final classifiers, correctly left randomly initialized for this binary task).

Loss: matches the official `gcnet-s_*_cityscapes-1024x1024.py` config exactly -- two `OhemCrossEntropy` terms (thres=0.9, min_kept=131072) on the two deep-supervision outputs (auxiliary head on stage-4 detail features, weight 0.4; main head on the fused final features, weight 1.0), ported from mmseg's `ohem_cross_entropy_loss.py` (itself derived from PIDNet's OHEM implementation).

Pretrained backbone: attempts to download the official GCNet-S Cityscapes checkpoint from Google Drive (verified alive and loadable ahead of time) via `gdown`, and loads backbone + head-feature weights (dropping only the 19-class final layers). Falls back to the paper's own from-scratch Kaiming initialization if that fails -- clearly logged either way, never silent.

Uses the same fixed size-stratified split, W&B logging, and D-FINE-matching summary.csv schema as every other baseline in this project.

Kaggle setup: attach **SmallDefectPreprocessing** as an input, enable Internet, and use a GPU.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time

RUN_NAME = 'RunN_gcnet_s_imgsz640'
MODEL_LABEL = 'GCNet-S'
WANDB_PROJECT = 'smallDefectDetection'
WANDB_RUN_NAME = f'{MODEL_LABEL}_segmentation'
IMG_SIZE = 640
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 15
LEARNING_RATE = 6e-5
WEIGHT_DECAY = 1e-2
NUM_WORKERS = 2
SEED = 42

# Official GCNet-S Cityscapes checkpoint (gyyang23/GCNet README), hosted on Google Drive.
# GCNet is trained from scratch on Cityscapes (no separate ImageNet-pretrained backbone release),
# so this is a segmentation-task-pretrained backbone, not an ImageNet-classification one -- if
# anything a more relevant transfer source for another dense-prediction task like ours.
GCNET_S_GDRIVE_ID = '1KersBP95k3b0AELiYlQ1rk4PKUmN-ueu'

DATASET_NAMES = ['DAGM', 'GC10-DET', 'KolektorSDD2', 'MPDD', 'MTD', 'Severstal', 'VisA']
SIZE_BUCKETS = ['small', 'medium', 'large']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
RUN_DIR = WORKING_ROOT / 'gcnet_runs' / RUN_NAME
FINAL_OUTPUT_DIR = WORKING_ROOT / 'final_outputs' / RUN_NAME
# Created unconditionally, right here -- not contingent on the pretrained-checkpoint try/except
# below reaching its mkdir line (which it only does if pip install + the Google Drive download
# both succeed first).
RUN_DIR.mkdir(parents=True, exist_ok=True)

print({'run': RUN_NAME, 'model': MODEL_LABEL, 'imgsz': IMG_SIZE, 'batch': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE})

In [ ]:
# Kaggle Internet must be enabled for this cell.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a Kaggle GPU before training.')

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True
print('Using device:', torch.cuda.get_device_name(0))


def wandb_login_anywhere():
    # Works on RunPod (env var) and Kaggle (Secrets add-on) without ever hardcoding the key.
    api_key = os.environ.get('WANDB_API_KEY')
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            api_key = None
    if api_key:
        wandb.login(key=api_key)
    else:
        wandb.login()


wandb_login_anywhere()
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        'model': MODEL_LABEL,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'seed': SEED,
    },
)

In [ ]:
# Find the normal SmallDefectPreprocessing Kaggle input automatically.
expected_datasets = set(DATASET_NAMES)
source_candidates = []

for root, dirs, _ in os.walk(KAGGLE_INPUT_ROOT):
    matches = expected_datasets.intersection(dirs)
    if len(matches) >= 5:
        source_candidates.append((len(matches), Path(root)))

if not source_candidates:
    raise FileNotFoundError(
        'Could not find the processed dataset. Attach SmallDefectPreprocessing as a Kaggle input.'
    )

source_candidates.sort(key=lambda item: (-item[0], len(str(item[1]))))
SOURCE_ROOT = source_candidates[0][1]
print('Using processed source:', SOURCE_ROOT)
print('Datasets:', sorted(path.name for path in SOURCE_ROOT.iterdir() if path.is_dir()))

In [ ]:
def index_files(directory, suffixes):
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    }

def candidate_stems(image_stem, target):
    base = image_stem.removesuffix('_defect')
    if target == 'mask':
        return [image_stem, image_stem.replace('_defect', '_mask'), base, base + '_mask', base + '_gt']
    return [image_stem, image_stem.replace('_defect', '_bbs'), base, base + '_bbs']

samples = []
missing = []

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        bucket_root = SOURCE_ROOT / dataset_name / size_bucket
        image_dir = bucket_root / 'images'
        mask_dir = bucket_root / 'masks'
        label_dir = bucket_root / 'labels_yolo'

        if not image_dir.exists() or not mask_dir.exists() or not label_dir.exists():
            missing.append((dataset_name, size_bucket, 'missing directory'))
            continue

        mask_index = index_files(mask_dir, IMAGE_EXTS)
        label_index = index_files(label_dir, {'.txt'})
        matched = 0

        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            mask_path = next((mask_index[stem] for stem in candidate_stems(image_path.stem, 'mask') if stem in mask_index), None)
            label_path = next((label_index[stem] for stem in candidate_stems(image_path.stem, 'box') if stem in label_index), None)

            if mask_path is None or label_path is None:
                missing.append((dataset_name, size_bucket, image_path.name))
                continue

            samples.append({
                'image_path': image_path,
                'mask_path': mask_path,
                'label_path': label_path,
                'dataset': dataset_name,
                'size': size_bucket,
                'stratum': dataset_name + '_' + size_bucket,
            })
            matched += 1

        print(f'{dataset_name}/{size_bucket}: {matched} matched')

print('Total matched:', len(samples))
print('Missing:', len(missing))
if len(samples) != 12670:
    raise RuntimeError(f'Expected 12,670 image-mask-label triplets, found {len(samples)}. First missing: {missing[:10]}')

In [ ]:
# Same fixed 70/15/15 stratified split as the other runs.
by_stratum = defaultdict(list)
for sample in samples:
    by_stratum[sample['stratum']].append(sample)

rng = random.Random(SEED)
train_samples, val_samples, test_samples = [], [], []
for _, group in sorted(by_stratum.items()):
    group = list(group)
    rng.shuffle(group)
    train_end = int(len(group) * 0.70)
    val_end = train_end + int(len(group) * 0.15)
    train_samples.extend(group[:train_end])
    val_samples.extend(group[train_end:val_end])
    test_samples.extend(group[val_end:])

rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)

test_sets = {
    'overall': test_samples,
    'small': [sample for sample in test_samples if sample['size'] == 'small'],
    'medium': [sample for sample in test_samples if sample['size'] == 'medium'],
    'large': [sample for sample in test_samples if sample['size'] == 'large'],
}

def print_counts(name, split):
    counts = Counter(sample['size'] for sample in split)
    print(f'{name}: total={len(split)}, small={counts["small"]}, medium={counts["medium"]}, large={counts["large"]}')

print_counts('Train', train_samples)
print_counts('Validation', val_samples)
for name, split in test_sets.items():
    print_counts('Test ' + name, split)

assert len(train_samples) == 8858
assert len(val_samples) == 1892
assert len(test_samples) == 1920

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def load_binary_mask(mask_path, target_size):
    mask = Image.open(mask_path).convert('L')
    if mask.size != target_size:
        mask = mask.resize(target_size, Image.Resampling.NEAREST)
    return (np.asarray(mask) > 0).astype(np.uint8)


class DefectMaskDataset(Dataset):
    def __init__(self, split_samples):
        self.samples = split_samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample['image_path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        mask = load_binary_mask(sample['mask_path'], (IMG_SIZE, IMG_SIZE))

        pixel_values = torch.from_numpy(np.asarray(image)).permute(2, 0, 1).float() / 255.0
        pixel_values = (pixel_values - IMAGENET_MEAN) / IMAGENET_STD
        labels = torch.from_numpy(mask).long()

        return {'pixel_values': pixel_values, 'labels': labels}


def make_loader(split_samples, shuffle=False):
    return DataLoader(
        DefectMaskDataset(split_samples),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )


train_loader = make_loader(train_samples, shuffle=True)
val_loader = make_loader(val_samples)
print('Train batches:', len(train_loader), 'Validation batches:', len(val_loader))

In [ ]:
# ---- Vendored from the official GCNet repo (gyyang23/GCNet), backbones/gcnet.py + ----
# ---- decode_heads/gcnet_head.py + utils/ppm.py's DAPPM. mmcv.ConvModule / mmengine.BaseModule ----
# ---- swapped for plain PyTorch equivalents; forward-pass math is unchanged. deploy=False (the ----
# ---- training-time multi-branch form) only -- switch_to_deploy/reparameterization is an ----
# ---- inference-speed optimization that doesn't affect any metric this project reports. ----

class ConvModule(nn.Module):
    """Matches real mmcv.cnn.ConvModule's attribute naming (.conv/.bn/.activate) and its
    'order' mechanism: order only changes forward() sequencing, not attribute names/nesting.
    When norm precedes conv in the order (DAPPM's pre-activation units), BN normalizes
    in_channels, not out_channels -- verified against the real GCNet-S checkpoint's shapes."""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, groups=1,
                 bias='auto', norm_cfg=None, act_cfg=dict(type='ReLU', inplace=True),
                 order=('conv', 'norm', 'act')):
        super().__init__()
        self.order = order
        if bias == 'auto':
            bias = norm_cfg is None
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding,
                              groups=groups, bias=bias)
        if norm_cfg is not None:
            norm_channels = in_channels if order.index('norm') < order.index('conv') else out_channels
            self.bn = nn.BatchNorm2d(norm_channels)
        else:
            self.bn = None
        self.activate = nn.ReLU(inplace=act_cfg.get('inplace', False)) if act_cfg is not None else None

    def forward(self, x):
        for layer in self.order:
            if layer == 'conv':
                x = self.conv(x)
            elif layer == 'norm' and self.bn is not None:
                x = self.bn(x)
            elif layer == 'act' and self.activate is not None:
                x = self.activate(x)
        return x


def build_norm_layer(norm_cfg, num_features):
    return 'bn', nn.BatchNorm2d(num_features)


def build_activation_layer(act_cfg):
    if act_cfg is None:
        return nn.Identity()
    return nn.ReLU(inplace=act_cfg.get('inplace', False))


def resize(input, size=None, scale_factor=None, mode='nearest', align_corners=None):
    return F.interpolate(input, size, scale_factor, mode, align_corners)


class DAPPM(nn.Module):
    """mmseg/models/utils/ppm.py -- only the piece GCNet's backbone actually calls."""

    def __init__(self, in_channels, branch_channels, out_channels, num_scales,
                 kernel_sizes=[5, 9, 17], strides=[2, 4, 8], paddings=[2, 4, 8],
                 norm_cfg=dict(type='BN'), act_cfg=dict(type='ReLU', inplace=True),
                 upsample_mode='bilinear'):
        super().__init__()
        self.num_scales = num_scales
        self.unsample_mode = upsample_mode

        def pre_act_conv(in_ch, out_ch, kernel_size=1, padding=0):
            return ConvModule(in_ch, out_ch, kernel_size, padding=padding, bias=False,
                              norm_cfg=norm_cfg, act_cfg=act_cfg, order=('norm', 'act', 'conv'))

        self.scales = nn.ModuleList([pre_act_conv(in_channels, branch_channels)])
        for i in range(1, num_scales - 1):
            self.scales.append(nn.Sequential(
                nn.AvgPool2d(kernel_size=kernel_sizes[i - 1], stride=strides[i - 1], padding=paddings[i - 1]),
                pre_act_conv(in_channels, branch_channels),
            ))
        self.scales.append(nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            pre_act_conv(in_channels, branch_channels),
        ))

        self.processes = nn.ModuleList([
            pre_act_conv(branch_channels, branch_channels, kernel_size=3, padding=1)
            for _ in range(num_scales - 1)
        ])
        self.compression = pre_act_conv(branch_channels * num_scales, out_channels)
        self.shortcut = pre_act_conv(in_channels, out_channels)

    def forward(self, inputs):
        feats = [self.scales[0](inputs)]
        for i in range(1, self.num_scales):
            feat_up = F.interpolate(self.scales[i](inputs), size=inputs.shape[2:], mode=self.unsample_mode)
            feats.append(self.processes[i - 1](feat_up + feats[i - 1]))
        return self.compression(torch.cat(feats, dim=1)) + self.shortcut(inputs)


class Block1x1(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, padding=0, bias=True, norm_cfg=dict(type='BN')):
        super().__init__()
        self.conv1 = ConvModule(in_channels, out_channels, 1, stride=stride, padding=padding, bias=bias, norm_cfg=norm_cfg, act_cfg=None)
        self.conv2 = ConvModule(out_channels, out_channels, 1, stride=1, padding=padding, bias=bias, norm_cfg=norm_cfg, act_cfg=None)

    def forward(self, x):
        return self.conv2(self.conv1(x))


class Block3x3(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, padding=0, bias=True, norm_cfg=dict(type='BN')):
        super().__init__()
        self.conv1 = ConvModule(in_channels, out_channels, 3, stride=stride, padding=padding, bias=bias, norm_cfg=norm_cfg, act_cfg=None)
        self.conv2 = ConvModule(out_channels, out_channels, 1, stride=1, padding=0, bias=bias, norm_cfg=norm_cfg, act_cfg=None)

    def forward(self, x):
        return self.conv2(self.conv1(x))


class GCBlock(nn.Module):
    """RepVGG-style multi-branch reparameterizable block: two 3x3 paths + one 1x1 path (+
    identity BN when shapes allow), summed and ReLU'd. Training-time (deploy=False) form only."""

    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1,
                 norm_cfg=dict(type='BN'), act_cfg=dict(type='ReLU', inplace=True), act=True):
        super().__init__()
        assert kernel_size == 3 and padding == 1
        padding_11 = padding - kernel_size // 2

        self.relu = build_activation_layer(act_cfg) if act else nn.Identity()
        if out_channels == in_channels and stride == 1:
            self.path_residual = build_norm_layer(norm_cfg, in_channels)[1]
        else:
            self.path_residual = None

        self.path_3x3_1 = Block3x3(in_channels, out_channels, stride=stride, padding=padding, bias=False, norm_cfg=norm_cfg)
        self.path_3x3_2 = Block3x3(in_channels, out_channels, stride=stride, padding=padding, bias=False, norm_cfg=norm_cfg)
        self.path_1x1 = Block1x1(in_channels, out_channels, stride=stride, padding=padding_11, bias=False, norm_cfg=norm_cfg)

    def forward(self, inputs):
        id_out = 0 if self.path_residual is None else self.path_residual(inputs)
        return self.relu(self.path_3x3_1(inputs) + self.path_3x3_2(inputs) + self.path_1x1(inputs) + id_out)


class GCNetBackbone(nn.Module):
    """GCNet-S: channels=32, ppm_channels=128, num_blocks_per_stage=[4,4,[5,4],[5,4],[2,2]]."""

    def __init__(self, in_channels=3, channels=32, ppm_channels=128,
                 num_blocks_per_stage=[4, 4, [5, 4], [5, 4], [2, 2]],
                 align_corners=False, norm_cfg=dict(type='BN'), act_cfg=dict(type='ReLU', inplace=True)):
        super().__init__()
        self.align_corners = align_corners

        def gc(in_ch, out_ch, stride=1, act=True):
            return GCBlock(in_ch, out_ch, stride=stride, norm_cfg=norm_cfg, act_cfg=act_cfg, act=act)

        self.stem = nn.Sequential(
            ConvModule(in_channels, channels, 3, stride=2, padding=1, norm_cfg=norm_cfg, act_cfg=act_cfg),
            ConvModule(channels, channels, 3, stride=2, padding=1, norm_cfg=norm_cfg, act_cfg=act_cfg),
            *[gc(channels, channels) for _ in range(num_blocks_per_stage[0])],
            gc(channels, channels * 2, stride=2),
            *[gc(channels * 2, channels * 2) for _ in range(num_blocks_per_stage[1] - 1)],
        )
        self.relu = build_activation_layer(act_cfg)

        self.semantic_branch_layers = nn.ModuleList([
            nn.Sequential(
                gc(channels * 2, channels * 4, stride=2),
                *[gc(channels * 4, channels * 4) for _ in range(num_blocks_per_stage[2][0] - 2)],
                gc(channels * 4, channels * 4, act=False),
            ),
            nn.Sequential(
                gc(channels * 4, channels * 8, stride=2),
                *[gc(channels * 8, channels * 8) for _ in range(num_blocks_per_stage[3][0] - 2)],
                gc(channels * 8, channels * 8, act=False),
            ),
            nn.Sequential(
                gc(channels * 8, channels * 16, stride=2),
                *[gc(channels * 16, channels * 16) for _ in range(num_blocks_per_stage[4][0] - 2)],
                gc(channels * 16, channels * 16, act=False),
            ),
        ])

        self.compression_1 = ConvModule(channels * 4, channels * 2, 1, norm_cfg=norm_cfg, act_cfg=None)
        self.down_1 = ConvModule(channels * 2, channels * 4, 3, stride=2, padding=1, norm_cfg=norm_cfg, act_cfg=None)

        self.compression_2 = ConvModule(channels * 8, channels * 2, 1, norm_cfg=norm_cfg, act_cfg=None)
        self.down_2 = nn.Sequential(
            ConvModule(channels * 2, channels * 4, 3, stride=2, padding=1, norm_cfg=norm_cfg, act_cfg=act_cfg),
            ConvModule(channels * 4, channels * 8, 3, stride=2, padding=1, norm_cfg=norm_cfg, act_cfg=None),
        )

        self.detail_branch_layers = nn.ModuleList([
            nn.Sequential(
                *[gc(channels * 2, channels * 2) for _ in range(num_blocks_per_stage[2][1] - 1)],
                gc(channels * 2, channels * 2, act=False),
            ),
            nn.Sequential(
                *[gc(channels * 2, channels * 2) for _ in range(num_blocks_per_stage[3][1] - 1)],
                gc(channels * 2, channels * 2, act=False),
            ),
            nn.Sequential(
                gc(channels * 2, channels * 4),
                *[gc(channels * 4, channels * 4) for _ in range(num_blocks_per_stage[4][1] - 2)],
                gc(channels * 4, channels * 4, act=False),
            ),
        ])

        self.spp = DAPPM(channels * 16, ppm_channels, channels * 4, num_scales=5, norm_cfg=norm_cfg, act_cfg=act_cfg)
        self.kaiming_init()

    def forward(self, x):
        out_size = (math.ceil(x.shape[-2] / 8), math.ceil(x.shape[-1] / 8))
        x = self.stem(x)

        x_s = self.semantic_branch_layers[0](x)
        x_d = self.detail_branch_layers[0](x)
        comp_c = self.compression_1(self.relu(x_s))
        x_s = x_s + self.down_1(self.relu(x_d))
        x_d = x_d + resize(comp_c, size=out_size, mode='bilinear', align_corners=self.align_corners)
        c4_feat = x_d.clone() if self.training else None

        x_s = self.semantic_branch_layers[1](self.relu(x_s))
        x_d = self.detail_branch_layers[1](self.relu(x_d))
        comp_c = self.compression_2(self.relu(x_s))
        x_s = x_s + self.down_2(self.relu(x_d))
        x_d = x_d + resize(comp_c, size=out_size, mode='bilinear', align_corners=self.align_corners)

        x_d = self.detail_branch_layers[2](self.relu(x_d))
        x_s = self.semantic_branch_layers[2](self.relu(x_s))
        x_s = self.spp(x_s)
        x_s = resize(x_s, size=out_size, mode='bilinear', align_corners=self.align_corners)

        return (c4_feat, x_d + x_s) if self.training else x_d + x_s

    def kaiming_init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)


class GCNetHead(nn.Module):
    """decode_heads/gcnet_head.py, with BaseDecodeHead's cls_seg (dropout + 1x1 conv) inlined."""

    def __init__(self, in_channels, channels, num_classes, dropout_ratio=0.0,
                 norm_cfg=dict(type='BN'), act_cfg=dict(type='ReLU', inplace=True)):
        super().__init__()

        def base_head(in_ch, ch):
            return nn.Sequential(
                nn.BatchNorm2d(in_ch),
                nn.ReLU(inplace=True),
                ConvModule(in_ch, ch, 3, padding=1, norm_cfg=norm_cfg, act_cfg=act_cfg),
            )

        self.head = base_head(in_channels, channels)
        self.aux_head_c4 = base_head(in_channels // 2, channels)
        self.aux_cls_seg_c4 = nn.Conv2d(channels, num_classes, kernel_size=1)

        self.dropout = nn.Dropout2d(dropout_ratio) if dropout_ratio > 0 else None
        self.conv_seg = nn.Conv2d(channels, num_classes, kernel_size=1)
        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def cls_seg(self, feat):
        if self.dropout is not None:
            feat = self.dropout(feat)
        return self.conv_seg(feat)

    def forward(self, inputs):
        if self.training:
            c4_feat, c6_feat = inputs
            c4_feat = self.aux_cls_seg_c4(self.aux_head_c4(c4_feat))
            c6_feat = self.cls_seg(self.head(c6_feat))
            return c4_feat, c6_feat
        else:
            return self.cls_seg(self.head(inputs))


class GCNet(nn.Module):
    """Top-level GCNet-S segmentor: backbone + head, upsampled to input resolution."""

    def __init__(self, num_classes=2, channels=32, ppm_channels=128,
                 num_blocks_per_stage=[4, 4, [5, 4], [5, 4], [2, 2]]):
        super().__init__()
        self.backbone = GCNetBackbone(channels=channels, ppm_channels=ppm_channels, num_blocks_per_stage=num_blocks_per_stage)
        self.decode_head = GCNetHead(in_channels=channels * 4, channels=64, num_classes=num_classes)

    def forward(self, x):
        in_size = x.shape[-2:]
        feats = self.backbone(x)
        if self.training:
            c4_logit, c6_logit = self.decode_head(feats)
            c4_logit = F.interpolate(c4_logit, size=in_size, mode='bilinear', align_corners=False)
            c6_logit = F.interpolate(c6_logit, size=in_size, mode='bilinear', align_corners=False)
            return c4_logit, c6_logit
        else:
            logit = self.decode_head(feats)
            return F.interpolate(logit, size=in_size, mode='bilinear', align_corners=False)


class OhemCrossEntropy(nn.Module):
    """mmseg/models/losses/ohem_cross_entropy_loss.py (itself adapted from PIDNet's OHEM CE).
    One instance per deep-supervision output, each with its own loss_weight -- matches the
    official gcnet-s config's loss_decode=[OhemCrossEntropy(...,loss_weight=0.4), OhemCrossEntropy(...,loss_weight=1.0)]."""

    def __init__(self, ignore_label=255, thres=0.9, min_kept=131072, loss_weight=1.0):
        super().__init__()
        self.thresh = thres
        self.min_kept = max(1, min_kept)
        self.ignore_label = ignore_label
        self.loss_weight = loss_weight

    def forward(self, score, target):
        pred = F.softmax(score, dim=1)
        pixel_losses = F.cross_entropy(score, target, ignore_index=self.ignore_label, reduction='none').contiguous().view(-1)
        mask = target.contiguous().view(-1) != self.ignore_label

        tmp_target = target.clone()
        tmp_target[tmp_target == self.ignore_label] = 0
        pred = pred.gather(1, tmp_target.unsqueeze(1))
        pred, ind = pred.contiguous().view(-1)[mask].contiguous().sort()
        if pred.numel() == 0:
            return score.new_tensor(0.0)
        min_value = pred[min(self.min_kept, pred.numel() - 1)]
        threshold = max(min_value, self.thresh)

        pixel_losses = pixel_losses[mask][ind]
        pixel_losses = pixel_losses[pred < threshold]
        return self.loss_weight * pixel_losses.mean()

In [ ]:
model = GCNet(num_classes=2).to(DEVICE)

pretrained_loaded = False
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'mmengine'], check=True)
    import gdown

    ckpt_path = RUN_DIR.parent / 'gcnet_s_cityscapes.pth'
    print('Downloading official GCNet-S Cityscapes checkpoint from Google Drive...')
    gdown.download(id=GCNET_S_GDRIVE_ID, output=str(ckpt_path), quiet=False)

    # mmengine is only needed to unpickle this checkpoint's metadata objects (message_hub,
    # param_schedulers) -- it's a lightweight pure-Python package, not the heavy mmcv/mmseg stack.
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint['state_dict']

    backbone_state = {k[len('backbone.'):]: v for k, v in state_dict.items() if k.startswith('backbone.')}
    head_state = {k[len('decode_head.'):]: v for k, v in state_dict.items() if k.startswith('decode_head.')}
    if not backbone_state:
        raise RuntimeError('Downloaded checkpoint had no backbone.* keys -- unexpected format.')

    missing_bb, unexpected_bb = model.backbone.load_state_dict(backbone_state, strict=False)
    print(f'Backbone: missing={len(missing_bb)}, unexpected={len(unexpected_bb)}')
    if missing_bb or unexpected_bb:
        # A real structural mismatch, not a class-count difference (backbone has no class-count
        # dependence at all) -- treat as a load failure rather than report a false positive.
        raise RuntimeError(f'Backbone key mismatch (missing={len(missing_bb)}, unexpected={len(unexpected_bb)}) -- treating as a load failure.')

    # Head: keep only class-count-independent layers, drop the 19-class Cityscapes final classifiers.
    model_head_dict = model.decode_head.state_dict()
    compatible_head_state = {k: v for k, v in head_state.items() if k in model_head_dict and v.shape == model_head_dict[k].shape}
    dropped_head_keys = set(head_state) - set(compatible_head_state)
    expected_dropped = {'conv_seg.weight', 'conv_seg.bias', 'aux_cls_seg_c4.weight', 'aux_cls_seg_c4.bias'}
    if dropped_head_keys != expected_dropped:
        raise RuntimeError(f'Unexpected set of dropped head keys: {sorted(dropped_head_keys)} -- checkpoint structure may not match the vendored head.')
    model.decode_head.load_state_dict(compatible_head_state, strict=False)
    print(f'Head: loaded {len(compatible_head_state)}/{len(head_state)} tensors (dropped the 4 Cityscapes 19-class final-layer tensors).')

    pretrained_loaded = True
except Exception as exc:
    print(f'Pretrained checkpoint download/load failed ({exc}); continuing with the paper\'s from-scratch Kaiming initialization. This is not silent -- pretrained_backbone_loaded=False is saved in run_metadata.json.')

wandb.config.update({'pretrained_backbone_loaded': pretrained_loaded})
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
@torch.inference_mode()
def evaluate(loader, measure_inference=False):
    model.eval()
    true_positive = false_positive = false_negative = 0
    inference_seconds = 0.0
    image_count = 0

    for batch in loader:
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        if measure_inference:
            torch.cuda.synchronize()
            start = time.perf_counter()
        logits = model(pixel_values)
        if measure_inference:
            torch.cuda.synchronize()
            inference_seconds += time.perf_counter() - start

        predictions = logits.argmax(dim=1)
        true_positive += int(((predictions == 1) & (labels == 1)).sum().item())
        false_positive += int(((predictions == 1) & (labels == 0)).sum().item())
        false_negative += int(((predictions == 0) & (labels == 1)).sum().item())
        image_count += labels.shape[0]

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    iou = true_positive / max(true_positive + false_positive + false_negative, 1)
    dice = 2 * true_positive / max(2 * true_positive + false_positive + false_negative, 1)

    return {
        'precision': precision,
        'recall': recall,
        'iou': iou,
        'dice': dice,
        # Pixel-level FPs per image, not per-instance -- this model has no discrete
        # "detections" to count false positives over, so it's normalized the honest
        # way: how many false-positive pixels on average per image in this split.
        'fp_per_image': false_positive / max(image_count, 1),
        'inference_time_ms_per_image': 1000 * inference_seconds / max(image_count, 1),
        'images': image_count,
    }


ohem_c4 = OhemCrossEntropy(thres=0.9, min_kept=131072, loss_weight=0.4)
ohem_c6 = OhemCrossEntropy(thres=0.9, min_kept=131072, loss_weight=1.0)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=True)

history = []
best_dice = -1.0
best_epoch = -1
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            c4_logit, c6_logit = model(pixel_values)
            loss = ohem_c4(c4_logit, labels) + ohem_c6(c6_logit, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item())

    val_metrics = evaluate(val_loader)
    row = {'epoch': epoch, 'train_loss': running_loss / len(train_loader), **val_metrics}
    history.append(row)
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | loss={row['train_loss']:.4f} | val Dice={row['dice']:.4f} | val IoU={row['iou']:.4f} | val Recall={row['recall']:.4f}")
    wandb.log(row, step=epoch)

    if row['dice'] > best_dice:
        best_dice = row['dice']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
        }, RUN_DIR / 'best_model.pt')
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(RUN_DIR / 'training_history.csv', index=False)
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation Dice was {best_dice:.4f} at epoch {best_epoch}.')
        break

wandb.summary['best_epoch'] = best_epoch
wandb.summary['best_validation_dice'] = best_dice
print('Best epoch:', best_epoch, 'Best validation Dice:', best_dice)

In [ ]:
# Load the best validation-Dice checkpoint and evaluate every fixed test subset.
checkpoint = torch.load(RUN_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_rows = []
for split_name, split_samples in test_sets.items():
    metrics = evaluate(make_loader(split_samples), measure_inference=True)
    test_rows.append({'split': split_name, **metrics})
    print(split_name, metrics)
    wandb.log({f'test_{split_name}_{key}': value for key, value in metrics.items()})

test_df = pd.DataFrame(test_rows)
display(test_df)

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(RUN_DIR / 'best_model.pt', FINAL_OUTPUT_DIR / 'best_model.pt')
shutil.copy2(RUN_DIR / 'training_history.csv', FINAL_OUTPUT_DIR / 'training_history.csv')
# Long-format, one row per split -- same shape as D-FINE's evaluation_metrics.csv.
test_df.to_csv(FINAL_OUTPUT_DIR / 'evaluation_metrics.csv', index=False)

by_split = {row['split']: row for row in test_rows}
overall = by_split['overall']

# Wide-format, one row per experiment -- matches the D-FINE/detection summary.csv schema
# exactly so this row can drop straight into the same combined results table. mAP columns
# are intentionally left blank: mAP is an instance-level metric and doesn't have a valid
# equivalent for a dense per-pixel segmenter like this one. Dice/IoU are kept as the honest
# segmentation-quality headline metric instead of a fabricated mAP.
summary_row = {
    'Experiment': RUN_NAME,
    'Model': MODEL_LABEL,
    'Batch': BATCH_SIZE,
    'Epochs': best_epoch,
    'mAP50': float('nan'),
    'mAP50_95': float('nan'),
    'Precision': overall['precision'],
    'Recall': overall['recall'],
    'mAP50_Small': float('nan'),
    'mAP50_Medium': float('nan'),
    'mAP50_Large': float('nan'),
    'Recall_Small': by_split['small']['recall'],
    'Recall_Medium': by_split['medium']['recall'],
    'Recall_Large': by_split['large']['recall'],
    'Inference_Time_ms': overall['inference_time_ms_per_image'],
    'FP_per_Image': overall['fp_per_image'],
    'Dice': overall['dice'],
    'IoU': overall['iou'],
    'Notes': f'{MODEL_LABEL}, vendored reparameterizable dual-branch backbone (no MMSegmentation dependency), deploy=False (training-time multi-branch form, no reparameterization fusion), pretrained_backbone_loaded={pretrained_loaded}, binary defect segmentation, fixed 640 split. mAP intentionally blank -- not a valid metric for dense semantic segmentation.',
}
summary_df = pd.DataFrame([summary_row])
display(summary_df)
summary_df.to_csv(FINAL_OUTPUT_DIR / 'summary.csv', index=False)

metadata = {
    'experiment': RUN_NAME,
    'model': MODEL_LABEL,
    'task': 'binary semantic defect segmentation',
    'pretrained_backbone_loaded': pretrained_loaded,
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'early_stopping_patience': PATIENCE,
    'best_epoch': best_epoch,
    'best_validation_dice': best_dice,
    'split_counts': {
        'train': len(train_samples), 'val': len(val_samples),
        'test': len(test_samples), 'test_small': len(test_sets['small']),
        'test_medium': len(test_sets['medium']), 'test_large': len(test_sets['large']),
    },
}
(FINAL_OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Saved model and all metrics to:', FINAL_OUTPUT_DIR)

wandb.finish()